# Day 5: Weather Database Integration, Automation, and Dashboard

## SparkCity Weather Feature

This notebook demonstrates the production workflow for Matthew's assigned weather feature:

- Validate the weather dataset before persistence
- Preview the guarded PostgreSQL ingestion pipeline
- Document the idempotent database loading strategy
- Demonstrate the scheduler-ready command-line interface
- Launch the interactive Streamlit weather dashboard
- Record operational monitoring and recovery procedures

> Database writes remain disabled in this notebook. The shared PostgreSQL loader uses `ON CONFLICT DO NOTHING`, but `--apply` should only be used after team approval.

In [41]:
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from sparkcityx.data_quality import validate_dataframe
from sparkcityx.loaders import load_dataset
from sparkcityx.weather_pipeline import run_weather_pipeline


ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

WEATHER_SOURCE = ROOT / "data" / "raw" / "weather_data.parquet"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("SparkCity-Weather-Day5")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Project root: {ROOT}")
print(f"Weather source: {WEATHER_SOURCE}")
print(f"Source exists: {WEATHER_SOURCE.exists()}")
print(f"Spark version: {spark.version}")

Project root: /Users/matthew/Projects/SparkCity_Capstone
Weather source: /Users/matthew/Projects/SparkCity_Capstone/data/raw/weather_data.parquet
Source exists: True
Spark version: 4.2.0


## 1. Pre-write Weather Validation

The production pipeline validates the complete weather schema, null counts, duplicate station/timestamp keys, timestamp compatibility, and measurement ranges before any database connection is used.

In [42]:
weather_df = load_dataset(spark, WEATHER_SOURCE)
validation_report = validate_dataframe(weather_df, "weather")

print(f"Weather rows: {validation_report['record_count']:,}")
print(f"Valid for loading: {validation_report['valid']}")
print(f"Missing columns: {validation_report['missing_columns']}")
print(f"Null counts: {validation_report['null_counts']}")
print(f"Duplicate rows: {validation_report['duplicate_count']}")
print(f"Range violations: {validation_report['range_violations']}")
print(
    "Timestamp violations:",
    validation_report["timestamp_violations"],
)

Weather rows: 36,000
Valid for loading: True
Missing columns: []
Null counts: {}
Duplicate rows: 0
Range violations: {}
Timestamp violations: {}


## 2. Guarded Weather Ingestion Preview

The weather pipeline defaults to preview mode. It loads and validates the source but does not open a PostgreSQL connection or insert records.

A database connection is required only when `apply=True`, preventing accidental writes during exploration or scheduled validation runs.

In [43]:
preview_result = run_weather_pipeline(
    spark,
    WEATHER_SOURCE,
    apply=False,
)

print("WEATHER INGESTION PREVIEW")
print("=" * 50)
print(f"Source: {preview_result.source}")
print(f"Validated rows: {preview_result.record_count:,}")
print(f"Valid: {preview_result.valid}")
print(f"Database applied: {preview_result.applied}")
print(f"Load result: {preview_result.load_result}")

WEATHER INGESTION PREVIEW
Source: /Users/matthew/Projects/SparkCity_Capstone/data/raw/weather_data.parquet
Validated rows: 36,000
Valid: True
Database applied: False
Load result: None


## 3. PostgreSQL Persistence Contract

Weather observations are stored in `sparkcity.weather_data`.

The composite primary key is:

- `station_id`
- `timestamp`

The loader validates the Spark DataFrame before opening its database transaction. Inserts use `ON CONFLICT DO NOTHING`, making repeated pipeline executions idempotent: existing station/timestamp records are preserved instead of overwritten.

In [44]:
from sparkcityx.database_load import get_load_config


load_config = get_load_config("weather")

print("WEATHER DATABASE CONTRACT")
print("=" * 50)
print(f"Dataset type: {load_config['dataset_type']}")
print(f"Target table: sparkcity.{load_config['table']}")
print(f"Primary key: {load_config['key']}")
print("Columns:")

for column in load_config["columns"]:
    print(f"  - {column}")

WEATHER DATABASE CONTRACT
Dataset type: weather
Target table: sparkcity.weather_data
Primary key: ['station_id', 'timestamp']
Columns:
  - station_id
  - timestamp
  - location_lat
  - location_lon
  - temperature
  - humidity
  - wind_speed
  - wind_direction
  - precipitation
  - pressure


## 4. Automated Pipeline and Scheduling

The registered pipeline command is:

```bash
uv run sparkcity-weather

In [45]:
preview_command = "uv run sparkcity-weather"
apply_command = "uv run sparkcity-weather --apply"

print("SCHEDULER-READY COMMANDS")
print("=" * 50)
print(f"Safe validation command: {preview_command}")
print(f"Approved database command: {apply_command}")
print()
print("Current notebook policy: PREVIEW ONLY")

SCHEDULER-READY COMMANDS
Safe validation command: uv run sparkcity-weather
Approved database command: uv run sparkcity-weather --apply

Current notebook policy: PREVIEW ONLY


## 5. Interactive Weather Dashboard Data

The Streamlit dashboard uses a reusable preparation layer to sort observations, derive date and hourly fields, and calculate operational summary metrics.

In [46]:
import pandas as pd

from sparkcityx.weather_dashboard import (
    prepare_weather_dashboard_data,
    summarize_weather_dashboard,
)


weather_pandas_df = pd.read_parquet(WEATHER_SOURCE)

dashboard_df = prepare_weather_dashboard_data(
    weather_pandas_df
)

dashboard_summary = summarize_weather_dashboard(
    dashboard_df
)

print("WEATHER DASHBOARD SUMMARY")
print("=" * 50)

for metric, value in dashboard_summary.items():
    if isinstance(value, float):
        print(f"{metric}: {value:,.2f}")
    else:
        print(f"{metric}: {value}")

WEATHER DASHBOARD SUMMARY
record_count: 36000
station_count: 1200
average_temperature: 52.06
average_humidity: 57.90
maximum_wind_speed: 23.98
total_precipitation: 4,249.82
average_pressure: 1,012.03
first_timestamp: 2025-01-01 00:00:00
last_timestamp: 2027-01-20 23:30:00


### Dashboard Launch Command

Run the interactive weather dashboard from the repository root:

```bash
uv run streamlit run dashboard/weather_app.py
```

The dashboard provides:

- Date and station filters
- Headline weather metrics
- Hourly temperature and humidity trends
- Wind speed and atmospheric pressure trends
- Precipitation totals
- A dataset-relative investigation queue
- Access to filtered weather observations

The investigation queue represents statistical extremes in the selected data. It does not claim to use official emergency-weather thresholds.

## 6. Monitoring and Recovery Strategy

The weather pipeline follows these operational safeguards:

1. Confirm the source file exists.
2. Load the Parquet dataset using the shared loader.
3. Validate schema, nulls, duplicates, timestamps, and measurement ranges.
4. Reject invalid data before opening a database transaction.
5. Require an explicit connection and `apply=True` before persistence.
6. Insert records in configurable batches.
7. Preserve existing primary-key rows with `ON CONFLICT DO NOTHING`.
8. Stop the Spark session even if pipeline execution fails.

Recommended recovery procedure:

- Review the pipeline error and validation report.
- Correct or quarantine invalid source records.
- Rerun `uv run sparkcity-weather` in preview mode.
- Confirm validation succeeds.
- Use `--apply` only after team approval.
- Rerunning an approved load is safe because existing keys are not replaced.

In [47]:
print("DAY 5 WEATHER PRODUCTION SUMMARY")
print("=" * 55)
print(f"Source records: {preview_result.record_count:,}")
print(f"Validation passed: {preview_result.valid}")
print(
    "Preview performed database write:",
    preview_result.applied,
)
print(f"Target table: sparkcity.{load_config['table']}")
print(f"Primary key: {load_config['key']}")
print("Safe pipeline command: uv run sparkcity-weather")
print(
    "Dashboard command: "
    "uv run streamlit run dashboard/weather_app.py"
)
print("Production status: READY FOR TEAM REVIEW")

DAY 5 WEATHER PRODUCTION SUMMARY
Source records: 36,000
Validation passed: True
Preview performed database write: False
Target table: sparkcity.weather_data
Primary key: ['station_id', 'timestamp']
Safe pipeline command: uv run sparkcity-weather
Dashboard command: uv run streamlit run dashboard/weather_app.py
Production status: READY FOR TEAM REVIEW


## Day 5 Conclusion

The weather feature now includes:

- A validated and idempotent PostgreSQL ingestion path
- A guarded preview-by-default production pipeline
- A scheduler-ready command-line interface
- An interactive Streamlit operations dashboard
- Automated tests covering preview, apply, invalid-data rejection, CLI behavior, and dashboard preparation
- Documented monitoring and recovery procedures

The shared database was not modified during this notebook demonstration.

In [48]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
